In [ ]:
import pandas as pd
import json
import os
import numpy as np

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [ ]:
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

def calc_retrieval_metrics(row, agent='baseline', k=3):
    # FIXED: Support both old and new result structures
    if agent in row and isinstance(row[agent], dict) and 'ids' in row[agent]:
        retrieved_ids = row[agent]['ids'][:k]
    elif f'{agent}_retrieved_ids' in row:
        retrieved_ids = row[f'{agent}_retrieved_ids'][:k]
    else:
        return pd.Series([0.0, 0.0, 0.0])

    gt_ids = [m['chunk_id'] for m in row['chunk_metadata']]
    gt_set = set(gt_ids)

    hits = [1 if doc_id in gt_set else 0 for doc_id in retrieved_ids]

    recall = sum(hits) / len(gt_set) if len(gt_set) > 0 else 0.0
    precision = sum(hits) / k if k > 0 else 0.0

    mrr = 0.0
    for i, hit in enumerate(hits):
        if hit == 1:
            mrr = 1.0 / (i + 1)
            break

    return pd.Series([precision, recall, mrr])
from config import Config


In [ ]:
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')
from config import Config

d:\Kuliah\SKRIPSI\code\playground\skripsi-playground-venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\jayaw\AppData\Local\Temp\ipykernel_42176\971595098.py:7: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\jayaw\AppData\Local\Temp\ipykernel_42176\971595098.py:7: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy


In [2]:
langchain_llm = ChatOpenAI(model="gpt-4o-mini")
langchain_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

evaluator_llm = LangchainLLMWrapper(langchain_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(langchain_embeddings)

C:\Users\jayaw\AppData\Local\Temp\ipykernel_42176\770287758.py:4: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(langchain_llm)
C:\Users\jayaw\AppData\Local\Temp\ipykernel_42176\770287758.py:5: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(langchain_embeddings)


In [ ]:
K_VALUE = Config.TOP_K
FILEPATH = f"knowledge-base/20260429_1829_rag_responses_dataset_k3.jsonl"

In [4]:
print("1. Memuat data hasil evaluasi...")
data_list = []
with open(FILEPATH, "r", encoding="utf-8") as f:
    for line in f:
        data_list.append(json.loads(line.strip()))

df = pd.DataFrame(data_list)
print(f"Total data dimuat: {len(df)} baris.\n")

# Ekstrak Kunci Jawaban ID (Ground Truth IDs) dari metadata
df['ground_truth_ids'] = df['chunk_metadata'].apply(lambda x: [meta['chunk_id'] for meta in x] if isinstance(x, list) else [])

# 1A. EKSTRAK ID RETRIEVAL DARI METADATA (MODIFIKASI BARU)
df['baseline_retrieved_ids'] = df['baseline_contexts_metadata'].apply(
    lambda x: [meta['chunk_id'] for meta in x] if isinstance(x, list) else []
)
df['fusion_retrieved_ids'] = df['fusion_contexts_metadata'].apply(
    lambda x: [meta['chunk_id'] for meta in x] if isinstance(x, list) else []
)

# 1B. BERSIHKAN TEKS JAWABAN DARI PROMPT (MODIFIKASI BARU)
def extract_final_answer(full_text):
    """Memotong prompt LLM dan hanya mengambil jawaban akhirnya saja"""
    if not isinstance(full_text, str):
        return ""
    
    # Token penanda di mana asisten mulai menjawab
    split_token = "<|im_start|>assistant\n"
    
    if split_token in full_text:
        # Ambil semua teks setelah token asisten
        return full_text.split(split_token)[-1].strip()
    return full_text.strip()

# Buat kolom baru berisi jawaban yang sudah bersih
df['clean_baseline_answer'] = df['baseline_answer'].apply(extract_final_answer)
df['clean_fusion_answer'] = df['fusion_answer'].apply(extract_final_answer)

1. Memuat data hasil evaluasi...
Total data dimuat: 50 baris.



In [5]:
df

,question,ground_truth,chunk_metadata,baseline_answer,baseline_contexts_metadata,baseline_retrieved_contexts,fusion_answer,fusion_contexts_metadata,fusion_retrieved_contexts,ground_truth_ids,baseline_retrieved_ids,fusion_retrieved_ids,clean_baseline_answer,clean_fusion_answer
0,Bagaimana pelaksanaan edukasi selama 15-30 men...,Pelaksanaan edukasi selama 15-30 menit bertuju...,[{'Header 2': 'RekapPengelolaan Penanganan Giz...,Human: <|im_start|>system\nAnda adalah asisten...,"[{'Header 2': '1. Tujuan', 'source': 'THORAX ...",[\n INFO KLINIS PASIEN:\n - Diagnosi...,Human: <|im_start|>system\n Anda adal...,"[{'Header 2': '3.5 Edukasi Gizi', 'source': '...",[\n INFO KLINIS PASIEN:\n - Diagnosi...,"[3449, 3427, 3424]","[3424, 3427, 3425]","[3442, 2548, 6882]",Edukasi selama 15–30 menit kepada orang tua pa...,Edukasi gizi yang dilaksanakan selama 15–30 me...
1,Apa hubungan antara pemberian makanan sehat da...,Pemberian makanan sehat berperan penting dalam...,[{'Header 2': 'RekapPengelolaan Penanganan Giz...,Human: <|im_start|>system\nAnda adalah asisten...,"[{'Header 2': '1. Tujuan', 'source': 'THORAX ...",[\n INFO KLINIS PASIEN:\n - Diagnosi...,Human: <|im_start|>system\n Anda adal...,"[{'Header 2': 'a. Riwayat Gizi Dahulu', 'sourc...",[\n INFO KLINIS PASIEN:\n - Diagnosi...,"[3449, 3427, 3424]","[3424, 3425, 3429]","[6443, 5435, 3444]",Tidak ada hubungan secara langsung antara pemb...,Pemberian makanan sehat memiliki hubungan pent...
2,Bagaimana hubungan antara kondisi prehipertens...,Pasien mengalami prehipertensi yang memerlukan...,[{'Header 2': '3.3 Monitoring dan Evaluasi Fis...,Human: <|im_start|>system\nAnda adalah asisten...,[{'Header 2': 'PENATALAKSANAAN GIZI PADA PASIE...,[\n INFO KLINIS PASIEN:\n - Diagnosi...,Human: <|im_start|>system\n Anda adal...,"[{'Header 2': '2.2.4 Fisik Klinis', 'source':...",[\n INFO KLINIS PASIEN:\n - Diagnosi...,"[4625, 4652, 4618]","[4599, 4653, 4597]","[4610, 4621, 4624]",Tidak ada informasi dalam konteks tentang reko...,Kondisi prehipertensi pasien menunjukkan bahwa...
3,Apa yang dapat disimpulkan tentang status gizi...,"Pasien tidak berisiko mengalami malnutrisi, ya...",[{'Header 2': '3.3 Monitoring dan Evaluasi Fis...,Human: <|im_start|>system\nAnda adalah asisten...,"[{'Header 2': 'Kesimpulan :', 'source': 'PASIE...",[\n INFO KLINIS PASIEN:\n - Diagnosi...,Human: <|im_start|>system\n Anda adal...,"[{'Header 2': 'BAB IV PEMBAHASAN', 'source': '...",[\n INFO KLINIS PASIEN:\n - Diagnosi...,"[4625, 4652, 4618]","[4653, 4646, 4645]","[1114, 4646, 1553]",Pasien memiliki indeks massa tubuh (IMT) sebes...,"Pasien dengan berat badan 52 kg, tinggi badan ..."
4,Apa hubungan antara penurunan kesadaran pasien...,Penurunan kesadaran pasien yang ditunjukkan de...,[{'Header 2': 'Lampiran 8. Dokumentasi Skrinin...,Human: <|im_start|>system\nAnda adalah asisten...,"[{'Header 2': 'B. Sasaran', 'source': 'Penuru...",[\n INFO KLINIS PASIEN:\n - Diagnosi...,Human: <|im_start|>system\n Anda adal...,[{'Header 2': 'B. Monitoring dan Evaluasi Peme...,[\n INFO KLINIS PASIEN:\n - Diagnosi...,"[4750, 4724, 4730]","[4708, 4748, 4659]","[4388, 2244, 4314]",Informasi tidak ditemukan. \n\nKonteks tidak ...,Informasi tentang pasien dengan jenis kelamin ...
5,Bagaimana kondisi bengkak pada mata pasien dap...,Kondisi bengkak pada mata pasien dapat dipenga...,[{'Header 2': 'Lampiran 8. Dokumentasi Skrinin...,Human: <|im_start|>system\nAnda adalah asisten...,"[{'Header 2': 'B. Prinsip Diet', 'source': 'P...",[\n INFO KLINIS PASIEN:\n - Diagnosi...,ERROR,[{'Header 2': 'B. Monitoring dan Evaluasi Peme...,[],"[4750, 4724, 4730]","[4702, 4708, 4709]","[4388, 2244, 4314]",Bengkak pada mata pasien dapat dipengaruhi ole...,ERROR
6,Apa tujuan pemberian diet yang diberikan kepad...,Tujuan pemberian diet adalah untuk meningkatka...,"[{'Header 2': 'A. Tujuan', 'source': 'WEDGE E...",Human: <|im_start|>system\nAnda adalah asisten...,"[{'Header 2': 'Lampiran 3. Media Edukasi', 's...",[\n INFO KLINIS PASIEN:\n - Diagnosi...,Human: <|im_st

In [6]:
def calc_retrieval_metrics(row, retrieved_col, k=3):
    retrieved_ids = row[retrieved_col][:k] # Potong hasil sesuai K
    ground_truth_ids = row['ground_truth_ids']
    
    gt_set = set(ground_truth_ids)
    
    # Deteksi mana yang "Hit" (Benar) di urutan top K
    hits = [1 if doc_id in gt_set else 0 for doc_id in retrieved_ids]
    
    # 1. Recall@K = (Jumlah Benar) / (Total Kunci Jawaban)
    recall = sum(hits) / len(gt_set) if len(gt_set) > 0 else 0.0
    
    # 2. Precision@K = (Jumlah Benar) / K
    precision = sum(hits) / k if k > 0 else 0.0
    
    # 3. MRR@K = 1 / (Ranking pertama yang benar)
    mrr = 0.0
    for i, hit in enumerate(hits):
        if hit == 1:
            mrr = 1.0 / (i + 1)
            break
            
    return pd.Series([precision, recall, mrr])

# Menetapkan Nilai K (Silakan ubah ke 3 atau 10 sesuai kebutuhan)
print(f"2. Menghitung Metrik Retrieval untuk K={K_VALUE}...")

# Hitung untuk Baseline
df[['baseline_Precision@K', 'baseline_Recall@K', 'baseline_MRR@K']] = df.apply(
    lambda row: calc_retrieval_metrics(row, 'baseline_retrieved_ids', k=K_VALUE), axis=1
)

# Hitung untuk Fusion
df[['fusion_Precision@K', 'fusion_Recall@K', 'fusion_MRR@K']] = df.apply(
    lambda row: calc_retrieval_metrics(row, 'fusion_retrieved_ids', k=K_VALUE), axis=1
)

2. Menghitung Metrik Retrieval untuk K=3...


In [7]:
print("\n3. Mempersiapkan data untuk evaluasi Ragas (LLM-as-a-Judge)...")

# Format Dataset untuk Ragas Baseline
dataset_baseline = Dataset.from_dict({
    "question": df["question"].tolist(),
    "answer": df["clean_baseline_answer"].tolist(), # MENGGUNAKAN JAWABAN BERSIH
    "contexts": df["baseline_retrieved_contexts"].tolist(),
    "ground_truth": df["ground_truth"].tolist()
})

# Format Dataset untuk Ragas Fusion
dataset_fusion = Dataset.from_dict({
    "question": df["question"].tolist(),
    "answer": df["clean_fusion_answer"].tolist(), # MENGGUNAKAN JAWABAN BERSIH
    "contexts": df["fusion_retrieved_contexts"].tolist(),
    "ground_truth": df["ground_truth"].tolist()
})

print("\n> Menjalankan Evaluasi Ragas untuk BASELINE...")
ragas_result_baseline = evaluate(
    dataset_baseline,
    metrics=[faithfulness, answer_relevancy],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)

print("\n> Menjalankan Evaluasi Ragas untuk FUSION...")
ragas_result_fusion = evaluate(
    dataset_fusion,
    metrics=[faithfulness, answer_relevancy],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)


3. Mempersiapkan data untuk evaluasi Ragas (LLM-as-a-Judge)...

> Menjalankan Evaluasi Ragas untuk BASELINE...


Evaluating:  23%|██▎       | 23/100 [01:37<04:09,  3.24s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  53%|█████▎    | 53/100 [03:04<01:54,  2.43s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  72%|███████▏  | 72/100 [04:11<01:43,  3.69s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  78%|███████▊  | 78/100 [04:20<00:49,  2.25s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  79%|███████▉  | 79/100 [04:28<01:07,  3.21s/it]LLM returned 1 generations instead of reques


> Menjalankan Evaluasi Ragas untuk FUSION...


Evaluating:   9%|▉         | 9/100 [00:45<06:35,  4.34s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  54%|█████▍    | 54/100 [03:01<02:14,  2.92s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 100/100 [05:34<00:00,  3.34s/it]


In [8]:
print("\n" + "="*50)
print(f"HASIL EVALUASI AKHIR (Top-K = {K_VALUE})")
print("="*50)

df_ragas_baseline = ragas_result_baseline.to_pandas()
df_ragas_fusion = ragas_result_fusion.to_pandas()

# 2. Hitung rata-rata (mean) dari kolom metrik, otomatis mengabaikan nilai NaN
baseline_faithfulness_mean = df_ragas_baseline['faithfulness'].mean()
baseline_answer_rel_mean = df_ragas_baseline['answer_relevancy'].mean()

fusion_faithfulness_mean = df_ragas_fusion['faithfulness'].mean()
fusion_answer_rel_mean = df_ragas_fusion['answer_relevancy'].mean()

summary_df = pd.DataFrame({
    "Metrik": [
        f"Context Recall@{K_VALUE}", 
        f"Context Precision@{K_VALUE}", 
        f"MRR@{K_VALUE}", 
        "Faithfulness (Ragas)", 
        "Answer Relevancy (Ragas)"
    ],
    "RAG Baseline": [
        df['baseline_Recall@K'].mean(),
        df['baseline_Precision@K'].mean(),
        df['baseline_MRR@K'].mean(),
        baseline_faithfulness_mean,
        baseline_answer_rel_mean
    ],
    "RAG Fusion": [
        df['fusion_Recall@K'].mean(),
        df['fusion_Precision@K'].mean(),
        df['fusion_MRR@K'].mean(),
        fusion_faithfulness_mean,
        fusion_answer_rel_mean
    ]
})

import IPython.display as display
display.display(summary_df)


HASIL EVALUASI AKHIR (Top-K = 3)


,Metrik,RAG Baseline,RAG Fusion
0,Context Recall@3,0.133333,0.080000
1,Context Precision@3,0.133333,0.080000
2,MRR@3,0.266667,0.190000
3,Faithfulness (Ragas),0.544991,0.673186
4,Answer Relevancy (Ragas),0.378343,0.404372


In [9]:
summary_df.to_csv(f'evaluation_@{K_VALUE}.csv')